# 30 — Validate `usdm_v4.ttl`

Parses the generated Turtle, runs SPARQL sanity counts, cross-checks NCIt
anchors against the source YAML, and appends a CSV row to
`../reports/validation.csv`.

Comparison against the prototype baseline (8,215 triples) is informational.
Deviation flags either source drift (likely benign, document in `docs/`) or
a generation bug (investigate).


## 1. Parse `usdm_v4.ttl`

Fail fast on syntax errors. Successful parse ⇒ the file is valid Turtle.


In [ ]:
import rdflib
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/20_generate_turtle.ipynb first."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
print(f"parsed {len(g)} triples from {TTL_PATH}")


## 2. Load source YAML for cross-checks

We re-parse `dataStructure.yml` here to compare what's in the graph
against what's in source.


In [ ]:
import yaml
import json

YAML_PATH = "../downloads/dataStructure.yml"
META_PATH = "../downloads/.fetch_meta.json"

with open(YAML_PATH) as f:
    SOURCE = yaml.safe_load(f)
with open(META_PATH) as f:
    META = json.load(f)

print(f"YAML  : {len(SOURCE)} top-level entries")
print(f"tag   : {META['ddf_ra_tag']}")
print(f"sha256: {META['sha256']}")


## 2b. Load USDM_CT bindings (for cross-checks)

Same loader as `20_generate_turtle.ipynb`. Builds `CT_BINDINGS` keyed by
`(entity, attr_name)` for verifying that every Y-row in source resolves
to the expected annotation triples in the graph.
Rows where `Inherited From` is non-null are skipped — the property IRI
exists only on the declaring class per v0's mechanical mapping, and the
inherited row duplicates the parent's binding.


In [ ]:
import openpyxl
import re
from pathlib import Path

CT_PATH      = "../downloads/USDM_CT.xlsx"
CT_META_PATH = "../downloads/.fetch_meta_ct.json"
if not Path(CT_PATH).exists():
    raise FileNotFoundError(
        f"{CT_PATH} missing — run notebooks/10_fetch_yaml.ipynb first."
    )

with open(CT_META_PATH) as f:
    CT_META = json.load(f)

CODE_TYPED_TARGETS = {"Code", "AliasCode", "ResponseCode"}
RE_URL_CCODE  = re.compile(r"/subset/ncit/(C\d+)\s*$")
RE_TEXT_CCODE = re.compile(r"\b(C\d+)\b")

wb = openpyxl.load_workbook(CT_PATH, read_only=True, data_only=True)
ws = wb["DDF Entities&Attributes"]
rows = list(ws.iter_rows(values_only=True))
header = list(rows[0])
i_entity = header.index("Entity Name")
i_attr   = header.index("Logical Data Model Name")
i_hvl    = header.index("Has Value List")
i_url    = header.index("Codelist URL")
i_inh    = header.index("Inherited From")

CT_BINDINGS = {}
for r in rows[1:]:
    if r[i_inh]:
        continue
    hvl = r[i_hvl]
    if not hvl or not str(hvl).startswith("Y"):
        continue
    ent = r[i_entity]
    att = r[i_attr]
    url = r[i_url]
    ccode_url, ccode_text = None, None
    if url:
        m = RE_URL_CCODE.search(str(url))
        if m: ccode_url = m.group(1)
    text_codes = RE_TEXT_CCODE.findall(str(hvl))
    if len(text_codes) > 1:
        raise ValueError(f"multiple C-codes in HVL for {ent}.{att}: {hvl!r}")
    if text_codes:
        ccode_text = text_codes[0]
    if ccode_url and ccode_text and ccode_url != ccode_text:
        raise ValueError(
            f"C-code mismatch for {ent}.{att}: url={ccode_url} text={ccode_text} hvl={hvl!r}"
        )
    ccode = ccode_url or ccode_text
    CT_BINDINGS[(ent, att)] = {
        "raw_hvl": str(hvl),
        "ccode":   ccode,
    }

n_total      = len(CT_BINDINGS)
n_structured = sum(1 for v in CT_BINDINGS.values() if v["ccode"])
n_freetext   = n_total - n_structured
print(f"CT_BINDINGS:   {n_total} Y-rows ({n_structured} structured, {n_freetext} free-text)")
print(f"ct tag:        {CT_META['ddf_ra_tag']}")
print(f"ct sha256:     {CT_META['sha256']}")


## 3. SPARQL sanity counts

Named classes (filter to IRIs to exclude `owl:unionOf` blank nodes).
Properties = `owl:DatatypeProperty` ∪ `owl:ObjectProperty`.


In [ ]:
QUERIES = {
    "triples": (
        "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"
    ),
    "classes_total": (
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a owl:Class . FILTER(isIRI(?c)) }"
    ),
    "classes_nci_anchored": (
        "PREFIX skos:<http://www.w3.org/2004/02/skos/core#> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { "
        "  ?c a owl:Class ; skos:exactMatch ?x . FILTER(isIRI(?c)) }"
    ),
    "properties_total": (
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { "
        "  { ?p a owl:DatatypeProperty } UNION { ?p a owl:ObjectProperty } }"
    ),
    "properties_nci_anchored": (
        "PREFIX skos:<http://www.w3.org/2004/02/skos/core#> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { "
        "  ?p skos:exactMatch ?x . "
        "  FILTER EXISTS { { ?p a owl:DatatypeProperty } UNION { ?p a owl:ObjectProperty } } }"
    ),
    "value_props": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p usdm:relationshipType 'Value' }"
    ),
    "ref_props": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p usdm:relationshipType 'Ref' }"
    ),
    "concrete_classes": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c usdm:modifier 'Concrete' . FILTER(isIRI(?c)) }"
    ),
    "abstract_classes": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c usdm:modifier 'Abstract' . FILTER(isIRI(?c)) }"
    ),
    "boundCodelist_triples": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(*) AS ?n) WHERE { ?p usdm:boundCodelist ?x }"
    ),
    "boundCodelistNote_triples": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(*) AS ?n) WHERE { ?p usdm:boundCodelistNote ?x }"
    ),
    "boundCodelist_nonNcit": (
        "PREFIX usdm:<https://w3id.org/cdisc/usdm/v4/> "
        "PREFIX ncit:<http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#> "
        "SELECT (COUNT(*) AS ?n) WHERE { "
        "  ?p usdm:boundCodelist ?x . "
        "  FILTER(!STRSTARTS(STR(?x), STR(ncit:))) }"
    ),
}

counts = {}
for label, sparql in QUERIES.items():
    counts[label] = int(list(g.query(sparql))[0][0])
    print(f"  {label:<25}: {counts[label]}")


## 4. Cross-check class-level NCIt anchors

For every source class with `NCI C-Code`, the graph must contain
`usdm:{Class} skos:exactMatch ncit:{C-Code}`. Mismatches are collected,
not raised — we want a complete report, not a first-failure exit.


In [ ]:
USDM = rdflib.Namespace("https://w3id.org/cdisc/usdm/v4/")
NCIT = rdflib.Namespace("http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#")
SKOS = rdflib.Namespace("http://www.w3.org/2004/02/skos/core#")

class_mismatches = []
for cname, entry in SOURCE.items():
    if not isinstance(entry, dict) or "NCI C-Code" not in entry:
        continue
    expected = NCIT[entry["NCI C-Code"]]
    subject = USDM[cname]
    if (subject, SKOS.exactMatch, expected) not in g:
        class_mismatches.append((cname, entry["NCI C-Code"]))

print(f"class NCIt cross-check: {len(class_mismatches)} mismatches")
for c, code in class_mismatches[:20]:
    print(f"  MISSING: usdm:{c} skos:exactMatch ncit:{code}")


## 5. Cross-check attribute-level NCIt anchors

Same check, scoped to non-inherited attributes. Property IRI is
`usdm:{ClassName}-{attributeName}`.


In [ ]:
attr_mismatches = []
for cname, entry in SOURCE.items():
    if not isinstance(entry, dict):
        continue
    for aname, attr in (entry.get("Attributes") or {}).items():
        if not isinstance(attr, dict) or "Inherited From" in attr:
            continue
        if "NCI C-Code" not in attr:
            continue
        expected = NCIT[attr["NCI C-Code"]]
        subject = USDM[f"{cname}-{aname}"]
        if (subject, SKOS.exactMatch, expected) not in g:
            attr_mismatches.append((cname, aname, attr["NCI C-Code"]))

print(f"attribute NCIt cross-check: {len(attr_mismatches)} mismatches")
for c, a, code in attr_mismatches[:20]:
    print(f"  MISSING: usdm:{c}-{a} skos:exactMatch ncit:{code}")


## 5b. Cross-check codelist bindings

For every Y-row in `CT_BINDINGS`, the graph must contain
`usdm:{Class}-{attr} usdm:boundCodelistNote "<raw>"` and, when a
structured codelist C-code is present, also
`usdm:{Class}-{attr} usdm:boundCodelist ncit:Cxxxxxx`.


In [ ]:
USDM_BOUND_CODELIST      = USDM["boundCodelist"]
USDM_BOUND_CODELIST_NOTE = USDM["boundCodelistNote"]

binding_mismatches = []
for (cname, aname), b in CT_BINDINGS.items():
    subject = USDM[f"{cname}-{aname}"]
    expected_note = rdflib.Literal(b["raw_hvl"])
    if (subject, USDM_BOUND_CODELIST_NOTE, expected_note) not in g:
        binding_mismatches.append((cname, aname, "boundCodelistNote", b["raw_hvl"]))
    if b["ccode"]:
        expected_ccode = NCIT[b["ccode"]]
        if (subject, USDM_BOUND_CODELIST, expected_ccode) not in g:
            binding_mismatches.append((cname, aname, "boundCodelist", b["ccode"]))

print(f"binding cross-check: {len(binding_mismatches)} mismatches")
for c, a, kind, val in binding_mismatches[:20]:
    print(f"  MISSING: usdm:{c}-{a} usdm:{kind} {val!r}")


## 6. Compare against prototype baseline

Baseline numbers come from prior prototype work against an earlier DDF-RA
snapshot. The triple-count baseline (8,215) reflects choices in that
prototype we cannot fully reconstruct — a residual delta of <300 is
expected and documented in `docs/known-gaps.md`. Structural counts
(classes, properties, anchored counts) should match exactly for tag
`v4.0.0` of DDF-RA.


In [ ]:
BASELINE = {
    "triples":                   8184,
    "classes_total":               86,
    "classes_nci_anchored":        84,
    "properties_total":           693,
    "properties_nci_anchored":    312,
    "boundCodelist_triples":       45,
    "boundCodelistNote_triples":   57,
}

print(f"{'metric':<25} {'baseline':>10} {'observed':>10} {'delta':>10}")
print("-" * 60)
for k, base in BASELINE.items():
    obs = counts[k]
    delta = obs - base
    print(f"{k:<25} {base:>10} {obs:>10} {delta:>+10}")


## 7. Append CSV row to `../reports/validation.csv`

One row per validation run, header on first write. History accumulates
in the file across runs.


In [ ]:
import csv
import datetime
from pathlib import Path

REPORT_PATH = "../reports/validation.csv"

row = {
    "timestamp_utc":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "ddf_ra_tag":              META["ddf_ra_tag"],
    "source_sha256":           META["sha256"],
    "triples":                 counts["triples"],
    "classes_total":           counts["classes_total"],
    "classes_nci_anchored":    counts["classes_nci_anchored"],
    "properties_total":        counts["properties_total"],
    "properties_nci_anchored": counts["properties_nci_anchored"],
    "value_props":             counts["value_props"],
    "ref_props":               counts["ref_props"],
    "concrete_classes":        counts["concrete_classes"],
    "abstract_classes":        counts["abstract_classes"],
    "class_mismatches":          len(class_mismatches),
    "attr_mismatches":           len(attr_mismatches),
    "ct_source_tag":             CT_META["ddf_ra_tag"],
    "ct_source_sha256":          CT_META["sha256"],
    "boundCodelist_triples":     counts["boundCodelist_triples"],
    "boundCodelistNote_triples": counts["boundCodelistNote_triples"],
    "binding_mismatches":        len(binding_mismatches),
}

Path(REPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
first_write = (not Path(REPORT_PATH).exists()) or Path(REPORT_PATH).stat().st_size == 0
with open(REPORT_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    if first_write:
        w.writeheader()
    w.writerow(row)

action = "created" if first_write else "appended to"
print(f"{action} {REPORT_PATH}")
for k, v in row.items():
    print(f"  {k:<25}: {v}")


## 8. Run summary

Empty mismatch lists ⇒ every NCI C-Code in source has a matching
`skos:exactMatch` triple in the graph. Non-empty lists are real bugs
to investigate.


In [ ]:
n_class_mm = len(class_mismatches)
n_attr_mm  = len(attr_mismatches)
n_bind_mm  = len(binding_mismatches)

if n_class_mm == 0 and n_attr_mm == 0 and n_bind_mm == 0:
    print("clean run: all source NCI anchors and codelist bindings present in graph")
else:
    print(f"class mismatches:     {n_class_mm}")
    print(f"attribute mismatches: {n_attr_mm}")
    print(f"binding mismatches:   {n_bind_mm}")
    print("see cells 4, 5, and 5b for details")
